## Identifies all tribtaries within the DEM that are >1km^2

In [9]:
import numpy as np
import rasterio
from rasterio.transform import xy
import pandas as pd
from scipy import ndimage

# ============== PARAMETERS TO ADJUST ==============
BASE_PATH = "/Users/Glong1/Desktop/Andes/Andes_watersheds/RapelRiver/BaseLevel/CoastalCordilleraTributaries/"
DEM_FILE = BASE_PATH + "coastal_cordillera_tributary_dem_utm_30m.tif"
AREA_FILE = BASE_PATH + "coastal_cordillera_tributary_area_utm30m"
FD_FILE = BASE_PATH + "coastal_cordillera_tributary_fd_utm30m"

# Drainage area thresholds (in square meters)
MIN_DRAINAGE_AREA = 2_000_000  # 2 km²
MAX_DRAINAGE_AREA = 500_000_000  # 500 km²

# Elevation threshold (in meters)
MAX_ELEVATION = 1000

# Spatial extent limits (UTM X coordinates)
MIN_X = 166836
MAX_X = 341552

# Ocean masking parameters
MIN_OCEAN_NEIGHBORS = 5  # out of 8 possible neighbors to classify as ocean

# Coastal outlet upstream adjustment
UPSTREAM_TRACE_DISTANCE = 100  # meters (~3-4 cells at 30m resolution)
MIN_OUTLET_ELEVATION = 0.1  # meters - move outlets upstream if below this

# Coastal outlet clustering - remove closely spaced outlets along coast
CLUSTER_DISTANCE = 1000  # meters - outlets closer than this will be clustered
APPLY_CLUSTERING = True  # set to False to disable clustering

# Limit number of outlets for testing (set to None for all)
MAX_OUTLETS = None

# Output file
OUTPUT_CSV = BASE_PATH + "tributary_outlets.csv"

# D8 flow direction encoding (ESRI/TauDEM style)
D8_DIRECTIONS = {
    1: (0, 1),    # East
    2: (1, 1),    # Southeast
    4: (1, 0),    # South
    8: (1, -1),   # Southwest
    16: (0, -1),  # West
    32: (-1, -1), # Northwest
    64: (-1, 0),  # North
    128: (-1, 1)  # Northeast
}

# ============== FUNCTIONS ==============

def load_raster(filepath):
    """Load a raster file and return data array and metadata."""
    with rasterio.open(filepath) as src:
        data = src.read(1)
        profile = src.profile
        transform = src.transform
        crs = src.crs
    return data, profile, transform, crs

def get_downstream_cell(row, col, fd_value):
    """Get the downstream cell coordinates based on D8 flow direction."""
    if fd_value in D8_DIRECTIONS:
        dr, dc = D8_DIRECTIONS[fd_value]
        return row + dr, col + dc
    return None, None

def get_upstream_cells(row, col, fd):
    """
    Find all cells that flow into the current cell.
    Returns list of (row, col) tuples.
    """
    upstream = []
    
    # Check all 8 neighbors
    for fd_val, (dr, dc) in D8_DIRECTIONS.items():
        neighbor_row = row - dr  # Reverse direction to find upstream
        neighbor_col = col - dc
        
        # Check if neighbor is in bounds
        if (0 <= neighbor_row < fd.shape[0] and 
            0 <= neighbor_col < fd.shape[1]):
            
            # Check if neighbor flows into current cell
            if fd[neighbor_row, neighbor_col] == fd_val:
                upstream.append((neighbor_row, neighbor_col))
    
    return upstream

def create_ocean_mask(elevation, min_ocean_neighbors=5):
    """
    Create a boolean mask identifying ocean cells.
    Ocean cells are those at elevation <= 0 AND surrounded by other cells at <= 0.
    
    Parameters:
    -----------
    elevation : np.array
        Elevation raster
    min_ocean_neighbors : int
        Minimum number of ocean neighbors (out of 8) to classify as ocean
    
    Returns:
    --------
    np.array : Boolean mask where True = ocean, False = land
    """
    ocean_mask = np.zeros(elevation.shape, dtype=bool)
    
    print("Creating ocean mask...")
    rows, cols = np.where(elevation <= 0)
    
    for idx, (row, col) in enumerate(zip(rows, cols)):
        ocean_count = 0
        total_valid = 0
        
        # Check all 8 neighbors
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                
                neighbor_row = row + dr
                neighbor_col = col + dc
                
                if (0 <= neighbor_row < elevation.shape[0] and 
                    0 <= neighbor_col < elevation.shape[1]):
                    
                    total_valid += 1
                    if elevation[neighbor_row, neighbor_col] <= 0:
                        ocean_count += 1
        
        # If most neighbors are ocean, mark as ocean
        if ocean_count >= min_ocean_neighbors:
            ocean_mask[row, col] = True
        
        if (idx + 1) % 10000 == 0:
            print(f"  Processed {idx+1}/{len(rows)} cells...")
    
    ocean_count = np.sum(ocean_mask)
    print(f"Ocean mask created: {ocean_count} cells identified as ocean")
    
    return ocean_mask

def trace_upstream_to_elevation(row, col, fd, elevation, min_elevation, max_cells=4):
    """
    Trace upstream from a cell until we find a cell above minimum elevation.
    
    Parameters:
    -----------
    row, col : int
        Starting cell coordinates
    fd : np.array
        Flow direction raster
    elevation : np.array
        Elevation raster
    min_elevation : float
        Target minimum elevation (meters)
    max_cells : int
        Maximum number of cells to trace upstream
    
    Returns:
    --------
    tuple : (row, col, elevation, cells_traced) or (None, None, None, 0) if not found
    """
    current_row, current_col = row, col
    
    for i in range(max_cells):
        # Check if current cell meets elevation threshold
        if elevation[current_row, current_col] > min_elevation:
            return current_row, current_col, elevation[current_row, current_col], i
        
        # Get upstream cells
        upstream_cells = get_upstream_cells(current_row, current_col, fd)
        
        if len(upstream_cells) == 0:
            # No upstream cells found
            return None, None, None, i
        
        # Follow the path with largest drainage area (main stem)
        # For simplicity, just take the first upstream cell
        # (in practice, they should all lead to similar elevations nearby)
        current_row, current_col = upstream_cells[0]
    
    # Reached max distance without finding elevated cell
    return None, None, None, max_cells

def cluster_outlets(outlets_list, distance_threshold=1000):
    """
    Cluster nearby outlets and keep only the one with largest drainage area from each cluster.
    """
    if len(outlets_list) == 0:
        return outlets_list
    
    coords = np.array([[o['utm_x'], o['utm_y']] for o in outlets_list])
    areas = np.array([o['drainage_area_m2'] for o in outlets_list])
    
    keep = np.ones(len(outlets_list), dtype=bool)
    
    for i in range(len(outlets_list)):
        if not keep[i]:
            continue
        
        distances = np.sqrt(np.sum((coords - coords[i])**2, axis=1))
        nearby = (distances < distance_threshold) & (distances > 0)
        
        if np.any(nearby):
            candidates = np.where((nearby) | (np.arange(len(outlets_list)) == i))[0]
            largest_idx = candidates[np.argmax(areas[candidates])]
            
            for idx in candidates:
                if idx != largest_idx:
                    keep[idx] = False
    
    filtered = [outlets_list[i] for i in range(len(outlets_list)) if keep[i]]
    print(f"  Clustering reduced {len(outlets_list)} outlets to {len(filtered)} (removed {len(outlets_list) - len(filtered)} clustered outlets)")
    
    return filtered

def is_tributary_outlet(row, col, area, fd, elevation, ocean_mask, 
                       min_area, max_area, max_elev):
    """
    Check if a cell is a tributary outlet using two rules:
    
    Rule 1: Tributary joining larger river
    - Drainage area within range (min_area to max_area)
    - Elevation below threshold
    - Flows into a cell with drainage area > max_area (larger system)
    - NOT in ocean mask
    
    Rule 2: Coastal river outlet
    - Drainage area within range (min_area to max_area)
    - NOT in ocean mask (is land)
    - Flows INTO ocean mask (downstream is ocean)
    
    Returns:
    --------
    tuple : (is_outlet, outlet_row, outlet_col) 
            - is_outlet: bool
            - outlet_row, outlet_col: adjusted position if moved upstream
    """
    current_area = area[row, col]
    current_elev = elevation[row, col]
    
    # Check drainage area is in range
    if not (min_area <= current_area <= max_area):
        return False, row, col
    
    # Must NOT be in ocean mask
    if ocean_mask[row, col]:
        return False, row, col
    
    # Get downstream cell
    fd_value = fd[row, col]
    down_row, down_col = get_downstream_cell(row, col, fd_value)
    
    # Check if downstream cell is valid (within bounds)
    downstream_valid = (down_row is not None and down_col is not None and
                       0 <= down_row < area.shape[0] and 
                       0 <= down_col < area.shape[1])
    
    # Rule 2: Coastal river outlet - flows into ocean mask
    if downstream_valid and ocean_mask[down_row, down_col]:
        # This is a coastal outlet
        # If elevation is too low, try to move upstream
        if current_elev <= MIN_OUTLET_ELEVATION:
            max_cells = int(UPSTREAM_TRACE_DISTANCE / 30) + 1  # 30m resolution
            new_row, new_col, new_elev, cells_traced = trace_upstream_to_elevation(
                row, col, fd, elevation, MIN_OUTLET_ELEVATION, max_cells
            )
            
            if new_row is not None:
                # Found elevated location upstream
                # Re-check criteria at new location
                new_area = area[new_row, new_col]
                if (min_area <= new_area <= max_area and 
                    not ocean_mask[new_row, new_col]):
                    return True, new_row, new_col
        
        # Either already elevated enough, or couldn't find better location
        return True, row, col
    
    # Rule 2 alternate: Flows out of raster bounds (also an outlet)
    if not downstream_valid:
        if current_elev <= MIN_OUTLET_ELEVATION:
            max_cells = int(UPSTREAM_TRACE_DISTANCE / 30) + 1
            new_row, new_col, new_elev, cells_traced = trace_upstream_to_elevation(
                row, col, fd, elevation, MIN_OUTLET_ELEVATION, max_cells
            )
            
            if new_row is not None:
                new_area = area[new_row, new_col]
                if (min_area <= new_area <= max_area and 
                    not ocean_mask[new_row, new_col]):
                    return True, new_row, new_col
        
        return True, row, col
    
    # Rule 1: Tributary joining larger river
    if current_elev > max_elev:
        return False, row, col
    
    downstream_area = area[down_row, down_col]
    
    # Tributary must be joining a stream larger than the max threshold
    if downstream_area > max_area:
        return True, row, col
    
    return False, row, col

# ============== MAIN PROCESSING ==============

print("Loading rasters...")
dem, dem_profile, dem_transform, dem_crs = load_raster(DEM_FILE)
area, area_profile, area_transform, area_crs = load_raster(AREA_FILE)
fd, fd_profile, fd_transform, fd_crs = load_raster(FD_FILE)

print(f"DEM shape: {dem.shape}")
print(f"Area range: {area.min():.2f} to {area.max():.2f} m²")
print(f"Elevation range: {dem.min():.2f} to {dem.max():.2f} m")

# Create ocean mask
ocean_mask = create_ocean_mask(dem, MIN_OCEAN_NEIGHBORS)

# Find candidate cells
print("\nSearching for tributary outlets...")
outlets = []
count = 0
moved_upstream_count = 0

# Iterate through all cells that are NOT in ocean mask
rows, cols = np.where(
    ~ocean_mask &
    (area >= MIN_DRAINAGE_AREA) & 
    (area <= MAX_DRAINAGE_AREA) & 
    (dem <= MAX_ELEVATION) &
    (dem != dem_profile['nodata']) &
    (area != area_profile['nodata'])
)

print(f"Found {len(rows)} candidate cells (not in ocean, within criteria)...")

for idx in range(len(rows)):
    row, col = rows[idx], cols[idx]
    
    # Get UTM coordinates for spatial filtering
    x, y = xy(dem_transform, row, col)
    
    # Check if within X bounds
    if not (MIN_X <= x <= MAX_X):
        continue
    
    is_outlet, outlet_row, outlet_col = is_tributary_outlet(
        row, col, area, fd, dem, ocean_mask,
        MIN_DRAINAGE_AREA, MAX_DRAINAGE_AREA, MAX_ELEVATION
    )
    
    if is_outlet:
        # Track if outlet was moved upstream
        if outlet_row != row or outlet_col != col:
            moved_upstream_count += 1
        
        # Get coordinates of final outlet position
        outlet_x, outlet_y = xy(dem_transform, outlet_row, outlet_col)
        
        outlets.append({
            'name': f'outlet_{count+1:04d}',
            'utm_x': outlet_x,
            'utm_y': outlet_y,
            'elevation_m': dem[outlet_row, outlet_col],
            'drainage_area_m2': area[outlet_row, outlet_col],
            'drainage_area_km2': area[outlet_row, outlet_col] / 1_000_000,
            'flow_direction': int(fd[outlet_row, outlet_col]),
            'row': outlet_row,
            'col': outlet_col,
            'moved_upstream': outlet_row != row or outlet_col != col
        })
        
        count += 1
        
        if MAX_OUTLETS and count >= MAX_OUTLETS:
            print(f"Reached maximum of {MAX_OUTLETS} outlets (for testing)")
            break
    
    # Progress update
    if (idx + 1) % 10000 == 0:
        print(f"  Processed {idx+1}/{len(rows)} candidates... Found {count} outlets so far")

print(f"\nFound {len(outlets)} tributary outlets")
print(f"  {moved_upstream_count} outlets were moved upstream from low elevation")

# Apply clustering if enabled
if APPLY_CLUSTERING and len(outlets) > 0:
    print(f"\nApplying clustering (distance threshold: {CLUSTER_DISTANCE}m)...")
    outlets = cluster_outlets(outlets, CLUSTER_DISTANCE)
    print(f"After clustering: {len(outlets)} outlets remain")

# Create DataFrame and save
df = pd.DataFrame(outlets)

if len(df) > 0:
    # Sort by drainage area
    df = df.sort_values('drainage_area_km2', ascending=False)
    
    # Save to CSV
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to: {OUTPUT_CSV}")
    
    # Print summary statistics
    print("\n=== SUMMARY ===")
    print(f"Total outlets: {len(df)}")
    print(f"Outlets moved upstream: {df['moved_upstream'].sum()}")
    print(f"Drainage area range: {df['drainage_area_km2'].min():.2f} - {df['drainage_area_km2'].max():.2f} km²")
    print(f"Elevation range: {df['elevation_m'].min():.2f} - {df['elevation_m'].max():.2f} m")
    
    # Flow direction distribution
    print(f"\nFlow direction distribution:")
    fd_counts = df['flow_direction'].value_counts().sort_index()
    fd_names = {1: 'E', 2: 'SE', 4: 'S', 8: 'SW', 16: 'W', 32: 'NW', 64: 'N', 128: 'NE'}
    for fd_val, count in fd_counts.items():
        print(f"  {fd_names.get(fd_val, fd_val)}: {count}")
    
    print("\nFirst 10 outlets:")
    print(df[['name', 'utm_x', 'utm_y', 'elevation_m', 'drainage_area_km2', 'moved_upstream']].head(10))
else:
    print("No outlets found matching criteria!")

Loading rasters...
DEM shape: (12142, 6336)
Area range: 748.05 to 28731806348.05 m²
Elevation range: -32768.00 to 2282.00 m
Creating ocean mask...
  Processed 10000/39314900 cells...
  Processed 20000/39314900 cells...
  Processed 30000/39314900 cells...
  Processed 40000/39314900 cells...
  Processed 50000/39314900 cells...
  Processed 60000/39314900 cells...
  Processed 70000/39314900 cells...
  Processed 80000/39314900 cells...
  Processed 90000/39314900 cells...
  Processed 100000/39314900 cells...
  Processed 110000/39314900 cells...
  Processed 120000/39314900 cells...
  Processed 130000/39314900 cells...
  Processed 140000/39314900 cells...
  Processed 150000/39314900 cells...
  Processed 160000/39314900 cells...
  Processed 170000/39314900 cells...
  Processed 180000/39314900 cells...
  Processed 190000/39314900 cells...
  Processed 200000/39314900 cells...
  Processed 210000/39314900 cells...
  Processed 220000/39314900 cells...
  Processed 230000/39314900 cells...
  Processed